In [11]:
import os
import pandas as pd
import numpy as np

In [12]:
lda_doc_scores_df = pd.read_csv("../data/output/LDA_DOC_SCORES.csv")
lda_loadings_df = pd.read_csv("../data/output/LDA_LOADINGS.csv", index_col=0)
lda_topic_means_df = pd.read_csv("../data/output/LDA_TOPIC_MEANS.csv")
lda_top_words_df = pd.read_csv("../data/output/LDA_TOP_WORDS.csv")

print("Theta shape:", lda_doc_scores_df.shape)
lda_doc_scores_df.head()

Theta shape: (31102, 5)


,T00,T01,T02,T03,T04
0,0.067189,0.73281,0.066667,0.066667,0.066667
1,0.200000,0.20000,0.200000,0.200000,0.200000
2,0.200000,0.20000,0.200000,0.200000,0.200000
3,0.399998,0.40000,0.066667,0.066667,0.066669
4,0.100000,0.10000,0.100000,0.100001,0.599999


In [13]:
delimiter = "|"

corpus = pd.read_csv("../data/output/CORPUS.csv", sep=delimiter)

corpus["Document"] = (
    corpus["book_number"].astype(str) + "_" +
    corpus["chapter"].astype(str) + "_" +
    corpus["verse"].astype(str)
)

doc_meta = (
    corpus
    .groupby("Document")
    .agg({"book_number": "first"})
    .reset_index()
)

book_key = pd.read_csv("../data/archive/key_english.csv")

doc_meta = doc_meta.merge(
    book_key[["b", "n"]],
    left_on="book_number",
    right_on="b",
    how="left"
)

doc_meta.rename(columns={"n": "Book"}, inplace=True)

theta_df = lda_doc_scores_df.copy()
theta_df["Document"] = doc_meta["Document"].values
theta_df["BookNum"] = doc_meta["book_number"].values
theta_df["Book"] = doc_meta["Book"].values

theta_df.head()

,T00,T01,T02,T03,T04,Document,BookNum,Book
0,0.067189,0.73281,0.066667,0.066667,0.066667,10_10_1,10,2 Samuel
1,0.200000,0.20000,0.200000,0.200000,0.200000,10_10_10,10,2 Samuel
2,0.200000,0.20000,0.200000,0.200000,0.200000,10_10_11,10,2 Samuel
3,0.399998,0.40000,0.066667,0.066667,0.066669,10_10_12,10,2 Samuel
4,0.100000,0.10000,0.100000,0.100001,0.599999,10_10_13,10,2 Samuel


In [14]:
topic_cols = [col for col in theta_df.columns if col.startswith("T")]

theta_df["Dominant_Topic"] = theta_df[topic_cols].idxmax(axis=1)

theta_df.head()

,T00,T01,T02,T03,T04,Document,BookNum,Book,Dominant_Topic
0,0.067189,0.73281,0.066667,0.066667,0.066667,10_10_1,10,2 Samuel,T01
1,0.200000,0.20000,0.200000,0.200000,0.200000,10_10_10,10,2 Samuel,T00
2,0.200000,0.20000,0.200000,0.200000,0.200000,10_10_11,10,2 Samuel,T00
3,0.399998,0.40000,0.066667,0.066667,0.066669,10_10_12,10,2 Samuel,T01
4,0.100000,0.10000,0.100000,0.100001,0.599999,10_10_13,10,2 Samuel,T04


In [15]:
theta_df[topic_cols].mean().sort_values(ascending=False)

T01    0.234657
T00    0.226774
T02    0.188174
T04    0.175792
T03    0.174603
dtype: float64

In [16]:
theta_df[topic_cols].sum(axis=1).head()

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float64

In [17]:
print(theta_df.isnull().sum())
print(theta_df.shape)

T00               0
T01               0
T02               0
T03               0
T04               0
Document          0
BookNum           0
Book              0
Dominant_Topic    0
dtype: int64
(31102, 9)


In [18]:
delimiter = "|"
print("Delimiter:", delimiter)

Delimiter: |


In [19]:
os.makedirs("../data/output", exist_ok=True)

theta_df.to_csv("../data/output/LDA_THETA_ENRICHED.csv", index=False)

print("LDA theta table saved.")

LDA theta table saved.
